# limeの結果を作成する


In [ ]:
matcher_names = ["wym"]
dataset_root_dir = "../../data/lemon/datasets"
#model_root_dir = "../../data/lemon/model"
model_root_dir = "../../data/wym/model"
out_root_dir = "../../data/experiments/11_eval/lime_result_wym_matcher"
dataset_names = [
    "structured_amazon_google",
    "structured_beer",
    "structured_dblp_acm",
    "structured_dblp_google_scholar",
    "structured_fodors_zagat",
    "structured_walmart_amazon",
    "structured_itunes_amazon",
    "dirty_dblp_acm",
    "dirty_dblp_google_scholar",
    "dirty_walmart_amazon",
    "dirty_itunes_amazon",
    "textual_abt_buy",
    "textual_company",
]

In [ ]:
TARGET_DATASET_ID = 1
TOP_N = 5
TARGET_MATCHER_ID = 0
START_DATA_IDX = None
END_DATA_IDX = None
GPU_ID = 1

In [ ]:
BATCH_SIZE = 512

In [ ]:
# スレッド数を制限
## これをしないと、他のプロセスが利用するCPUがなくなってしまう
import os

THREAD_NUM = 5

# NumPy (OpenBLAS, MKL 等) が使用するスレッド数を制限
os.environ["OMP_NUM_THREADS"] = str(THREAD_NUM)
os.environ["MKL_NUM_THREADS"] = str(THREAD_NUM)
os.environ["OPENBLAS_NUM_THREADS"] = str(THREAD_NUM)

import numpy as np
import torch

# PyTorch のスレッド数制限 (演算用)
torch.set_num_threads(THREAD_NUM)
# PyTorch のスレッド数制限 (DataLoader 等のインタロップ用)
torch.set_num_interop_threads(THREAD_NUM)


In [ ]:
# torchモジュールの読み込み前に、利用できるGPUを指定しておく
## これをやらないと、システム内のＧＰＵすべてを利用してしまう
import os

os.environ["CUDA_VISIBLE_DEVICES"] = f"{GPU_ID}"

import torch

print("CUDA =", torch.cuda.is_available())
print("CUDA DEVICES =", torch.cuda.device_count())
print("CUDA CURRENT DEVICE_ID = ", torch.cuda.current_device())

In [ ]:
# transformers の tokenizer を並列実行で呼び出すか（dead lockしてしまう）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Set Random Seeds and Reproducibility
import random

import numpy as np


def set_seed(seed: int):
    """
    Helper function for reproducible behavior to set the seed in ``random``, ``numpy``, ``torch``
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

### WYM へのデータ変換

In [ ]:
import pandas as pd
import unicodedata

from pine.dataset import load_dataset
from lemon.utils.datasets import SplittedDataset


def remove_accents(input_str: str) -> str:
    """
    Unicode のアクセント記号を削除して基本的な a-z に変換
    Args:
        input_str (str): 入力文字列
    Returns:
        str: アクセント記号を削除した文字列
    """
    # Unicode 正規化 (NFKD) によって分解
    normalized_str = unicodedata.normalize("NFKD", input_str)
    # 分解されたアクセント記号を除去
    return "".join(c for c in normalized_str if not unicodedata.combining(c))


def normalize_str(df: pd.DataFrame) -> pd.DataFrame:
    """
    文字列の正規化を行う。ウムラウト系を削除。大文字小文字を統一する。
    Args:
        df (pd.DataFrame): 文字列を含むDataFrame
    Returns:
        pd.DataFrame: 正規化されたDataFrame
    """

    # 置換関数を定義
    def replace_chars(value):
        if isinstance(value, str):  # 文字列のみ処理
            value = remove_accents(value).lower()
        return value

    # 各列ごとに処理し、元の型を維持
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].apply(replace_chars).astype("string")
        else:
            df[col] = df[col]  # 非文字列型はそのまま
    return df


def convert_dataset_to_wym(
    records_left: pd.DataFrame,
    records_right: pd.DataFrame,
    record_id_pairs: pd.DataFrame,
    labels: pd.Series = None,
) -> pd.DataFrame:
    """
    lemonのデータセットをWYMの入力形式に変換する
    WWMの入力形式は以下のDatarrameを返す。
    id,left_id,right_id,label,left_*(左側のカラム), right_右側のカラム

    Args:
        records_left (pd.DataFrame): 左側のレコード
        records_right (pd.DataFrame): 右側のレコード
        record_id_pairs (pd.DataFrame): レコードIDのペア
        labels (pd.DataFrame, optional): ラベル. Defaults to None.
    Returns:
        pd.DataFrame: WYMの入力形式
    """
    # record_leftとrecord_rightの内容のうち文字列を変更する
    records_left = normalize_str(records_left)
    records_right = normalize_str(records_right)

    # カラム名を変更
    df = record_id_pairs.rename(columns={"a.rid": "left_id", "b.rid": "right_id"})
    # ラベルがあれば結合
    if labels is not None:
        df = pd.merge(df, labels.astype(int), left_index=True, right_index=True)

    # レコードIDのペアを結合
    df = pd.merge(
        df,
        records_left.add_prefix("left_"),
        left_on="left_id",
        right_index=True,
    )
    df = pd.merge(
        df, records_right.add_prefix("right_"), left_on="right_id", right_index=True
    )

    # インデックスをリセットしpidをidに変換する
    df = df.sort_index().reset_index().rename(columns={"pid": "id"})

    return df


def test_convert_dataset_to_wym():
    dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset: SplittedDataset = load_dataset(dataset_name, dataset_root_dir)
    df = convert_dataset_to_wym(
        dataset.test.records.a,
        dataset.test.records.b,
        dataset.test.record_id_pairs,
        dataset.test.labels,
    )
    display(df.head())
    print("DATA SIZE =", len(df))
    print(df.dtypes)


test_convert_dataset_to_wym()


In [ ]:
from typing import Callable, List

import pandas as pd
from pine.entity import EntityPair, Entity


def convert_entity_pairs_to_wym(entity_pairs: List[EntityPair]) -> pd.DataFrame:
    """
    EntityPair形式をWYMの入力形式に変換する

    Args:
        entity_pairs (List[EntityPair]): EntityPairのリスト
    Returns:
        pd.DataFrame: WYMの入力形式
    """
    df_lefts = []
    df_rights = []
    df_pairs = []
    for idx, entity_pair in enumerate(entity_pairs):
        df_left, df_right = entity_pair.to_dataframe()
        # idを振りなおしながら追加
        df_left.index = [idx]
        df_right.index = [len(entity_pairs) + idx]
        df_lefts.append(df_left)
        df_rights.append(df_right)
        df_pairs.append(
            pd.DataFrame(
                {
                    "pid": [idx],
                    "a.rid": [df_left.index[0]],
                    "b.rid": [df_right.index[0]],
                }
            )
        )
    df_lefts = pd.concat(df_lefts)
    df_rights = pd.concat(df_rights)
    df_pairs = pd.concat(df_pairs).set_index("pid")
    record_pair = convert_dataset_to_wym(df_lefts, df_rights, df_pairs)
    return record_pair


def test_convert_entity_pairs_to_wym():
    dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset: SplittedDataset = load_dataset(dataset_name, dataset_root_dir)

    entity_pairs = []
    for pid, l_id, r_id in dataset.train.record_id_pairs[:10].itertuples():
        entity_pair = EntityPair(
            Entity.from_dataframe(dataset.train.records.a.loc[[l_id]]),
            Entity.from_dataframe(dataset.train.records.b.loc[[r_id]]),
        )
        entity_pairs.append(entity_pair)
    df = convert_entity_pairs_to_wym(entity_pairs)
    display(df)
    print("DATA SIZE =", len(df))
    print(df.dtypes)
    assert len(entity_pairs) == 10
    return


test_convert_entity_pairs_to_wym()


### WYM マッチャー関数

In [ ]:
from typing import Callable, List
import os
import pickle


import torch
from wym.wym import Wym
from wym.Net import NetAccoppiate, DatasetAccoppiate
from pine.entity import EntityPair, Entity

import warnings

# SettingWithCopyWarning を無視する設定
#warnings.simplefilter(action="ignore", category=FutureWarning)
#warnings.simplefilter(
#    action="ignore", category=UserWarning
#)  # これで SettingWithCopyWarning を無視


# モンキーパッチ
## メソッドの変更
def new_relevance_score(self, word_pairs, emb_pairs):
    self.word_pair_model.eval()
    self.word_pair_model.to(self.device)
    # data_loader = self.train_data_loader
    # data_loader.__init__(word_pairs, emb_pairs)
    data_loader = DatasetAccoppiate(word_pairs, emb_pairs)
    word_pair_corrected = data_loader.word_pairs_corrected
    with torch.no_grad():
        word_pair_corrected["pred"] = (
            self.word_pair_model(data_loader.X.to(self.device)).cpu().detach().numpy()
        )
    return word_pair_corrected


Wym.relevance_score = new_relevance_score


## メンバー変数の変更
def modify_wym_member(wym: Wym) -> Wym:
    # word_pair_model の追加
    tmp_path = os.path.join(wym.model_files_path, "net.pickle")
    best_model = NetAccoppiate()
    best_model.load_state_dict(
        torch.load(tmp_path, map_location=torch.device(wym.device))
    )
    wym.word_pair_model = best_model

    # best_linear_model_data の追加
    tmp_path = os.path.join(wym.model_files_path, "linear_model.pickle")
    with open(tmp_path, "rb") as file:
        model_data = pickle.load(file)
    wym.best_linear_model_data = model_data

    # word_embeddings の verbode をFalseに変更
    wym.we.verbose = False

    return wym


def make_wym_matcher_func_core(
    model_dir: str, data_columns: List[str], exclule_columns: List[str]
) -> Callable[[List[EntityPair], bool], np.array]:
    """Wym用のマッチャー関数を作成する"""
    df_empty = pd.DataFrame(columns=data_columns)
    # モデルの読み込み
    wym = Wym(
        df=df_empty,
        exclude_attrs=exclule_columns,
        model_files_path=model_dir,
        batch_size=BATCH_SIZE,
        reset_networks=False,
        verbose=False,
    )
    wym = modify_wym_member(wym)

    def proba_func(
        entity_pairs: List[EntityPair], expand_axis: bool = True
    ) -> np.array:
        # EntityPairをWYMのDataFrameに変換
        df = convert_entity_pairs_to_wym(entity_pairs)

        # 空のデータしかない場合、エラーとなるので、空のデータしかない場合は0を返す
        is_empty = df[wym.columns_to_use].replace("", np.nan).isna().all().all()
        if is_empty:
            return np.array([0] * len(entity_pairs))

        # マッチング確率の計算
        scores = wym.predict(df[wym.columns_to_use].copy(), return_data=False)

        # スコアを規格化 0.0 - 1.0 を -1.0 - 1.0 にする
        scores = 2 * scores - 1.0
        if expand_axis:
            # limeでは、1データに複数のラベルの結果がある場合が想定されているため、一軸増増やしたデータを作成
            return scores[:, np.newaxis]
        return scores

    return proba_func


def make_wym_matcher_func(
    dataset_name: str, model_root_dir: str, dataset_root_dir: str
) -> Callable[[List[EntityPair], bool], np.array]:
    dataset: SplittedDataset = load_dataset(dataset_name, dataset_root_dir)
    df_wfm = convert_dataset_to_wym(
        dataset.test.records.a,
        dataset.test.records.b,
        dataset.test.record_id_pairs,
        dataset.test.labels,
    )
    data_columns = df_wfm.columns.tolist().copy()
    exclude_columns = ["id", "left_id", "right_id", "label"]
    model_dir = os.path.join(model_root_dir, dataset_name)
    return make_wym_matcher_func_core(model_dir, data_columns, exclude_columns)


def test_proba_fn():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    proba_fn = make_wym_matcher_func(
        target_dataset_name, model_root_dir, dataset_root_dir
    )

    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    entity_pairs = []
    for idx in range(5):
        pair_id = dataset.test.record_id_pairs.iloc[idx : idx + 1]
        entity_l = Entity.from_dataframe(
            dataset.test.records.a[
                pair_id.iloc[0].loc["a.rid"] : pair_id.iloc[0]["a.rid"] + 1
            ]
        )
        entity_r = Entity.from_dataframe(
            dataset.test.records.b[
                pair_id.iloc[0].loc["b.rid"] : pair_id.iloc[0]["b.rid"] + 1
            ]
        )
        entity_pairs.append(EntityPair(entity_l, entity_r))
    entity_l = entity_pairs[0].entity_l.make_entity_by_deleting_segments(
        range(entity_pairs[0].entity_l.segment_size())
    )
    entity_r = entity_pairs[0].entity_r.make_entity_by_deleting_segments(
        range(entity_pairs[0].entity_r.segment_size())
    )
    empty_entity_pair = EntityPair(entity_l, entity_r)
    entity_pairs.append(empty_entity_pair)

    scores = proba_fn(entity_pairs, False)

    for pair, score in zip(entity_pairs, scores):
        display(pair.entity_l.to_dataframe())
        display(pair.entity_r.to_dataframe())
        print(score)
    empty_scores = proba_fn([empty_entity_pair], False)
    display(empty_entity_pair.to_dataframe())
    display(empty_entity_pair.to_dataframe())
    print(empty_scores)

    scores_single = [proba_fn([pair], False)[0] for pair in entity_pairs]

    np.testing.assert_array_almost_equal(scores, scores_single, decimal=5)


test_proba_fn()

## LIME 結果作成

In [ ]:
import pathlib
import pickle
from typing import List

import tqdm

from pine.dataset import load_dataset
from pine.entity import Entity, EntityPair
from pine.explainer import AttributionScore
from pine.explainer.lime_explainer import make_explanation, kernel


def save_lime_results(
    dataset,
    matcher_func,
    out_dir,
    save_step,
    top_n,
    start_data_idx=None,
    end_data_idx=None,
    sample_num=None,
):
    if sample_num is None or sample_num > len(dataset.test.record_id_pairs):
        sample_num = len(dataset.test.record_id_pairs)
        test_sampled = dataset.test.record_id_pairs
    else:
        test_sampled = dataset.test.record_id_pairs.sample(n=sample_num, random_state=0)

    for i in tqdm.tqdm(range(0, sample_num, save_step)):
        # 開始idxよりも前ならskip
        if (
            start_data_idx is not None
            and i < int(start_data_idx / save_step) * save_step
        ):
            print("skip step {}".format(i))
            continue
        # 終了idxよりも後ろならskip
        if (
            end_data_idx is not None
            and int((end_data_idx - 1) / save_step) * save_step < i
        ):
            print("skip step {}".format(i))
            continue

        lime_results = {}
        out_file_path = pathlib.Path(out_dir) / f"{i}.pickle"

        if out_file_path.exists():
            # ファイルがあればデータを読み込む
            with out_file_path.open("rb") as f:
                lime_results = pickle.load(f)

        for data_count, (pid, l_id, r_id) in tqdm.tqdm(
            enumerate(
                test_sampled[i : i + save_step].itertuples()
            ),
            total=save_step,
            leave=False,
        ):
            # 開始idxよりも前ならskip
            if start_data_idx is not None and i + data_count < start_data_idx:
                print("skip {}".format(i + data_count))
                continue
            # 終了idxよりも後ろならskip
            if end_data_idx is not None and end_data_idx - 1 < i + data_count:
                print("skip {}".format(i + data_count))
                continue
            if pid not in lime_results:
                lime_results[pid] = {}

            entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
            entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])

            # オリジナルがなければ作る
            if (None, None) not in lime_results[pid]:
                ret = make_explanation(
                    EntityPair(entity_l, entity_r), matcher_func, kernel, None, 0, False
                )
                lime_results[pid][(None, None)] = ret

            # # TOP K Attribution score (attribution score >0 and attribution score 降順)を削除
            # attribution_score_list_l: List[AttributionScore] = lime_results[pid][
            #     (None, None)
            # ][0]
            # attribution_score_list_r: List[AttributionScore] = lime_results[pid][
            #     (None, None)
            # ][1]
            # attribution_score_list_l = sorted(
            #     attribution_score_list_l, key=lambda x: x.score, reverse=True
            # )
            # attribution_score_list_r = sorted(
            #     attribution_score_list_r, key=lambda x: x.score, reverse=True
            # )
            # del_idx_ls = []
            # del_idx_rs = []
            # # top n target idx
            # for attribution_score in attribution_score_list_l[:top_n]:
            #     if attribution_score.score <= 0:
            #         break
            #     del_idx_ls.append(attribution_score.index)
            # for attribution_score in attribution_score_list_r[:top_n]:
            #     if attribution_score.score <= 0:
            #         break
            #     del_idx_rs.append(attribution_score.index)
            # # bottom n target idx
            # for attribution_score in attribution_score_list_l[::-1][:top_n]:
            #     if attribution_score.score >= 0:
            #         break
            #     del_idx_ls.append(attribution_score.index)
            # for attribution_score in attribution_score_list_r[::-1][:top_n]:
            #     if attribution_score.score >= 0:
            #         break
            #     del_idx_rs.append(attribution_score.index)

            # # in entity_l
            # for del_idx_l in del_idx_ls:
            #     # 未作成なら作る
            #     if (del_idx_l, None) not in lime_results[pid]:
            #         entity_l_del = entity_l.make_entity_by_deleting_segments(
            #             [del_idx_l]
            #         )
            #         ret = make_explanation(
            #             EntityPair(entity_l_del, entity_r), matcher_func, kernel, None, 0, False
            #         )
            #         lime_results[pid][(del_idx_l, None)] = ret
            # # in entity_r
            # for del_idx_r in del_idx_rs:
            #     # 未作成なら作る
            #     if (None, del_idx_r) not in lime_results[pid]:
            #         entity_r_del = entity_r.make_entity_by_deleting_segments(
            #             [del_idx_r]
            #         )
            #         ret = make_explanation(
            #             EntityPair(entity_l, entity_r_del), matcher_func, kernel, None, 0, False
            #         )
            #         lime_results[pid][(None, del_idx_r)] = ret

        with out_file_path.open("wb") as f:
           pickle.dump(lime_results, f)
        print("save {}".format(str(out_file_path)))
    return True

In [ ]:
import gc

save_step = 100
sample_num = None

target_dataset_name = dataset_names[TARGET_DATASET_ID]
dataset = load_dataset(target_dataset_name, dataset_root_dir)
for target_matcher_name in matcher_names:
    print("=======================")
    print(target_dataset_name, target_matcher_name)
    print("=======================")
    if TARGET_MATCHER_ID is not None and target_matcher_name != matcher_names[TARGET_MATCHER_ID]:
        print("SKIP. Because TARGET_MATCHER_NAME={}".format(matcher_names[TARGET_MATCHER_ID]))
        continue
    # if target_matcher_name == "magellan":
    #     matcher_func = make_magellan_matcher_func(
    #         target_dataset_name, model_root_dir
    #     )
    # elif target_matcher_name == "bert_mini":
    #     matcher_func = make_transformer_matcher_func(
    #         target_dataset_name, model_root_dir
    #     )
    if target_matcher_name == "wym":
        matcher_func = make_wym_matcher_func(
            target_dataset_name, model_root_dir, dataset_root_dir
        )
    else:
        raise ValueError("Invalid target_matcher_name {}".format(target_matcher_name))
    out_dir_path = (
        pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
    )
    out_dir_path.mkdir(parents=True, exist_ok=True)
    save_lime_results(dataset, matcher_func, out_dir_path, save_step, TOP_N, START_DATA_IDX, END_DATA_IDX, sample_num)
    del matcher_func
    gc.collect()